# DeepSeek official-workload preview

This notebook renders the new SWE-Bench Lite + Terminal-Bench rows through the
DeepSeek Harness. It uses the preview snapshot while the full catalog run
continues, and writes figures outside the paper image directory for review.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results' / 'deepseek_harness'
source = RESULTS / 'preview_official_tasks_raw.csv'
if not source.exists():
    source = RESULTS / 'official_tasks_raw.csv'
df = pd.read_csv(source)
df = df[df['harness_backend'].eq('deepseek_harness')].copy()
df['task_label'] = df['suite'].str.upper() + ': ' + df['task'].str.replace('__', '/', regex=False)
mode_order = ['causal', 'temporal_checkpoint', 'whole_branch_abort']
mode_labels = {'causal': 'Causal', 'temporal_checkpoint': 'Temporal checkpoint', 'whole_branch_abort': 'Whole-branch abort'}
palette = {'causal': '#c00000', 'temporal_checkpoint': '#e78129', 'whole_branch_abort': '#3f4b5a'}
plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Nimbus Roman', 'Times New Roman', 'Times'], 'font.size': 8, 'pdf.fonttype': 42})
out = RESULTS / 'preview_figures'
out.mkdir(parents=True, exist_ok=True)
tasks = list(dict.fromkeys(df['task_label']))
x = range(len(tasks))

# Figure 1: actual DeepSeek token usage for each official task and policy.
fig, ax = plt.subplots(figsize=(8.0, 3.2), dpi=220)
for mode in mode_order:
    view = df[df['mode'].eq(mode)].set_index('task_label').reindex(tasks)
    ax.plot(x, view['total_tokens'], marker='o', linewidth=1.1, markersize=3.5, color=palette[mode], label=mode_labels[mode])
ax.set_xticks(list(x), tasks, rotation=28, ha='right', fontsize=7)
ax.set_ylabel('Total tokens (DeepSeek Flash)')
ax.set_xlabel('Official workload task')
ax.grid(axis='y', linestyle=':', linewidth=.45, color='#b9c1c9')
ax.legend(frameon=True, fontsize=7, ncol=3)
fig.tight_layout()
fig.savefig(out / 'FIG-DeepSeek-Official-Tokens.pdf', bbox_inches='tight')
fig.savefig(out / 'FIG-DeepSeek-Official-Tokens.png', dpi=260, bbox_inches='tight')
plt.close(fig)

# Figure 2: verifier success and independent-work retention.
fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.0), dpi=220, sharey=True)
for ax, metric, title in zip(axes, ['success', 'independent_retained'], ['Verifier success', 'Independent work retained']):
    for mode in mode_order:
        view = df[df['mode'].eq(mode)].set_index('task_label').reindex(tasks)
        values = view[metric].astype(float).to_numpy()
        ax.plot(x, values, marker='o', linewidth=1.0, markersize=3.5, color=palette[mode], label=mode_labels[mode])
    ax.set_title(title, fontsize=9)
    ax.set_xticks(list(x), tasks, rotation=28, ha='right', fontsize=6.5)
    ax.set_ylim(-.05, 1.05)
    ax.grid(axis='y', linestyle=':', linewidth=.45, color='#b9c1c9')
axes[0].set_ylabel('Rate')
axes[1].legend(frameon=True, fontsize=6.5)
fig.tight_layout()
fig.savefig(out / 'FIG-DeepSeek-Official-Retention.pdf', bbox_inches='tight')
fig.savefig(out / 'FIG-DeepSeek-Official-Retention.png', dpi=260, bbox_inches='tight')
plt.close(fig)

# Figure 3: workload scale and tail usage.
scale_order = {'short': 0, 'medium': 1, 'long': 2}
df['scale_id'] = df['scale'].map(scale_order)
view = df.groupby(['suite', 'scale', 'scale_id', 'mode'], as_index=False)[['total_tokens', 'total_tokens']].mean()
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), dpi=220, sharey=False)
for mode in mode_order:
    for suite, group in view[view['mode'].eq(mode)].groupby('suite'):
        group = group.sort_values('scale_id')
        label = f'{suite.upper()} / {mode_labels[mode]}'
        axes[0].plot(group['scale_id'], group['total_tokens'], marker='o', linewidth=1.0, color=palette[mode], label=label)
        axes[1].plot(group['scale_id'], group['total_tokens'], marker='o', linewidth=1.0, color=palette[mode], label=label)
for ax in axes:
    ax.set_xticks([0, 1, 2], ['short', 'medium', 'long'])
    ax.set_xlabel('Official task scale')
    ax.grid(axis='y', linestyle=':', linewidth=.45, color='#b9c1c9')
axes[0].set_title('Mean tokens', fontsize=9)
axes[1].set_title('p95/p99 require repeated runs', fontsize=9)
axes[1].text(.5, .5, 'Preview uses one repeat\n(full run records p95/p99)', ha='center', va='center', transform=axes[1].transAxes, fontsize=8, color='#66727e')
axes[1].set_yticks([])
axes[0].set_ylabel('Tokens')
axes[0].legend(frameon=True, fontsize=5.5)
fig.tight_layout()
fig.savefig(out / 'FIG-DeepSeek-Official-Scaling.pdf', bbox_inches='tight')
fig.savefig(out / 'FIG-DeepSeek-Official-Scaling.png', dpi=260, bbox_inches='tight')
plt.close(fig)
print('source:', source)
print('rows:', len(df), 'tasks:', len(tasks), 'usage sources:', sorted(df['usage_source'].dropna().unique()))
print('figures:', *sorted(str(p) for p in out.glob('FIG-DeepSeek-Official-*')), sep='\n')
